# Fine-tuning Mini LLM Local — Google Colab

Treina (QLoRA) um modelo pequeno (Gemma 2 2B / Llama 3.2 1B / Phi-3 mini) no dataset de suporte técnico do projeto e exporta um arquivo **`.gguf`** pronto para o app C# (LLamaSharp).

**Antes de começar:** `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU (T4)**.

Fluxo: `dataset .jsonl` → QLoRA (`finetune.py`) → merge → GGUF (`convert_to_gguf.py`) → download → pasta `models/` local.

## 1. Conferir a GPU
Deve aparecer uma Tesla T4 com ~15 GB. Se der erro, ative a GPU no menu (passo acima).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clonar o repositório e instalar dependências

In [ ]:
!git clone https://github.com/ricardosmdemello/Mini-LLM-Local-C-Sharp
%cd Mini-LLM-Local-C-Sharp/training
!pip install -q -r requirements.txt

## 3. Login no Hugging Face
Necessário para modelos *gated* (Gemma e Llama oficiais). Crie um token em https://huggingface.co/settings/tokens e aceite os termos do modelo na página dele.

> Para **Phi-3** (licença MIT, sem gate) você pode pular esta célula.

In [ ]:
from huggingface_hub import login
login()  # cole seu token quando solicitado

## 4. Escolher o modelo base e treinar
Edite `BASE_MODEL` se quiser outro modelo. Opções comuns:
- `google/gemma-2-2b-it`  (gated)
- `meta-llama/Llama-3.2-1B-Instruct`  (gated)
- `microsoft/Phi-3-mini-4k-instruct`  (MIT, sem gate)

O dataset de exemplo tem poucos exemplos (apenas para validar o pipeline). Para um modelo útil, use algumas centenas de exemplos no mesmo formato `instruction/input/output`.

In [ ]:
BASE_MODEL = "google/gemma-2-2b-it"
OUTPUT_DIR = "./output/meu-modelo-suporte"
DATASET    = "../data/support-dataset.jsonl"

!python finetune.py \
    --base-model $BASE_MODEL \
    --dataset $DATASET \
    --output-dir $OUTPUT_DIR \
    --epochs 3 \
    --merge

## 5. Compilar o llama.cpp (para a conversão GGUF)
Roda uma vez por sessão (leva ~2-3 min).

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF >/dev/null && cmake --build llama.cpp/build --config Release -j 2 >/dev/null
print('llama.cpp pronto')

## 6. Converter o modelo mesclado para GGUF (quantizado Q4_K_M)

In [ ]:
import os
os.environ["LLAMA_CPP_DIR"] = os.path.abspath("llama.cpp")
GGUF_OUT = "meu-modelo-suporte-Q4_K_M.gguf"

!python convert_to_gguf.py \
    --model-dir $OUTPUT_DIR/merged \
    --outfile $GGUF_OUT \
    --quantize Q4_K_M

!ls -lh $GGUF_OUT

## 7. Baixar o .gguf
Faça o download e coloque o arquivo na pasta **`models/`** do projeto local. Ele aparecerá automaticamente na lista de modelos do app (console e web).

In [ ]:
from google.colab import files
files.download(GGUF_OUT)

---
**Pronto!** No projeto local:
```bash
# coloque o .gguf em models/ e rode:
dotnet run --project src/MiniLlmLocal.Console   # ou MiniLlmLocal.Web
```
Selecione o modelo recém-treinado no Chat e converse — ele responderá no estilo/conteúdo do seu dataset.